# Geography Chatbot with LangGraphA small command-line style chatbot for students. It answers geography questions and politely declines other topics.Before running the notebook, add an API key named `OPENAI_API_KEY` to Colab Secrets (the key icon in the left sidebar) and enable notebook access to that secret.

## 1. Install the required packagesRun this cell once in your Colab runtime.

In [ ]:
!pip -q install langgraph langchain langchain-openai

## 2. Load your API key from Colab SecretsThe key is read at runtime and stored only in this runtime environment. Never paste it into a code cell or save it in the notebook.

In [ ]:
import osfrom google.colab import userdata# Read the key from Colab Secrets and make it available to LangChain.os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 3. Define the chatbot graphThe graph has one `chatbot` node. The `add_messages` reducer appends each returned assistant reply to the conversation state.

In [ ]:
from typing import Annotated, TypedDictfrom langchain.chat_models import init_chat_modelfrom langchain_core.messages import AnyMessage, HumanMessage, SystemMessagefrom langgraph.graph import END, START, StateGraphfrom langgraph.graph.message import add_messages# The system message keeps the chatbot focused on geography.SYSTEM = SystemMessage(    content="You are a helpful chatbot for students. Answer questions about geography only. If asked about another topic, politely explain that you can help with geography questions instead.")# add_messages keeps the conversation history in the graph state.class ChatState(TypedDict):    messages: Annotated[list[AnyMessage], add_messages]# Choose the requested OpenAI chat model.model = init_chat_model("openai:gpt-6-luna")# The node sends the system instruction and conversation to the model.def chatbot(state: ChatState):    reply = model.invoke([SYSTEM] + state["messages"])    return {"messages": [reply]}# Build and compile a one-node graph without a checkpointer.builder = StateGraph(ChatState)builder.add_node("chatbot", chatbot)builder.add_edge(START, "chatbot")builder.add_edge("chatbot", END)graph = builder.compile()

## 4. Chat with the botEnter a geography question and press Return. Type `stop` to end the chat. The loop runs outside the graph and keeps the conversation history between turns.

In [ ]:
history = []while True:    text = input("You (type 'stop' to finish): ")    if text.strip().lower() == "stop":        print("Goodbye!")        break    result = graph.invoke({"messages": history + [HumanMessage(content=text)]})    history = result["messages"]    print("Bot:", history[-1].content)